# LightGBM 5-Fold Cross-Validation for Rome LST

This notebook is fully self-contained. It:

1. loads `rome_dataset_features_target_with_focal.csv`;
2. excludes `LST` and the coordinate columns from the predictors;
3. runs shuffled 5-fold cross-validation with the provided LightGBM hyperparameters;
4. calculates R², MAE and MAPE for train and validation in every fold;
5. saves fold metrics, summary statistics and feature importances as CSV files.

The split is random. Therefore, these results measure random pixel interpolation within Rome, not generalisation to geographically unseen areas.

In [1]:
from pathlib import Path
from time import perf_counter
import importlib.util

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.model_selection import KFold

if importlib.util.find_spec("lightgbm") is None:
    raise ImportError("LightGBM is not installed. Run: %pip install lightgbm")

from lightgbm import LGBMRegressor

RANDOM_STATE = 42
N_SPLITS = 5
TARGET = "LST"
COORDINATE_COLUMNS = ["x_utm", "y_utm", "lon", "lat"]
OUTPUT_DIR = Path("model_outputs/lightgbm_5fold_cv")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_PARAMS = {
    "random_state": 42,
    "verbosity": -1,
    "n_jobs": -1,
    "n_estimators": 300,
    "num_leaves": 24,
    "min_child_samples": 128,
    "reg_lambda": 0.03857720932178909,
    "reg_alpha": 0.21598801171228973,
    "subsample": 0.8347063033889262,
    "colsample_bytree": 0.8759561732083417,
    "learning_rate": 0.01879471459020519,
}

print("Output directory:", OUTPUT_DIR.resolve())
print("LightGBM parameters:", BEST_PARAMS)

Output directory: /Users/simon/Downloads/earth_obs/urban_heat_island_analysis_EODA/model_outputs/lightgbm_5fold_cv
LightGBM parameters: {'random_state': 42, 'verbosity': -1, 'n_jobs': -1, 'n_estimators': 300, 'num_leaves': 24, 'min_child_samples': 128, 'reg_lambda': 0.03857720932178909, 'reg_alpha': 0.21598801171228973, 'subsample': 0.8347063033889262, 'colsample_bytree': 0.8759561732083417, 'learning_rate': 0.01879471459020519}


In [2]:
def find_dataset():
    candidates = [
        Path("/Users/simon/Downloads/rome_dataset_features_target_with_focal.csv"),
        Path("../../rome_dataset_features_target_with_focal.csv"),
        Path("../rome_dataset_features_target_with_focal.csv"),
        Path("rome_dataset_features_target_with_focal.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "rome_dataset_features_target_with_focal.csv was not found. "
        "Place it next to the notebook or update the candidate paths."
    )


def infer_features(dataframe):
    excluded = {TARGET, *COORDINATE_COLUMNS}
    features = [
        column
        for column in dataframe.select_dtypes(include=np.number).columns
        if column not in excluded
    ]
    if not features:
        raise ValueError("No numeric predictor columns were found.")
    return features


def calculate_metrics(y_true, y_pred):
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return {
        "R2": float(r2_score(y_true, y_pred)),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "MAPE": float(mape),
        "MAPE_percent": float(100 * mape),
    }


DATA_PATH = find_dataset()
print("Dataset:", DATA_PATH)

df = pd.read_csv(DATA_PATH)
FEATURES = infer_features(df)

required_columns = FEATURES + [TARGET]
missing_rows = int(df[required_columns].isna().any(axis=1).sum())
if missing_rows:
    print(f"Removing {missing_rows:,} rows containing missing model values.")
    df = df.dropna(subset=required_columns).reset_index(drop=True)

X = df[FEATURES]
y = df[TARGET]

print(f"Rows: {len(df):,}")
print(f"Features: {len(FEATURES)}")
print("Coordinates excluded from predictors:", COORDINATE_COLUMNS)
display(df[FEATURES + [TARGET]].describe().T)

Dataset: /Users/simon/Downloads/rome_dataset_features_target_with_focal.csv


Rows: 677,735
Features: 34
Coordinates excluded from predictors: ['x_utm', 'y_utm', 'lon', 'lat']


,count,mean,std,min,25%,50%,75%,max
VV,677735.0,-8.530368,3.541054,-2.029688e+01,-10.950043,-9.134044,-6.590623,26.590615
VH,677735.0,-15.346800,3.238331,-2.832052e+01,-17.272451,-15.447249,-13.548643,20.142515
VV_VH_diff,677735.0,6.816432,2.149448,-1.181730e+01,5.706516,6.352742,7.413758,37.066587
VH_VV_ratio,677735.0,0.226369,0.079943,1.964904e-04,0.181395,0.231593,0.268750,0.505406
RFDI,677735.0,0.636313,0.113579,-8.765129e-01,0.576355,0.623913,0.692915,0.999607
SAR_urban,677735.0,-1.770998,4.587536,-1.717360e+01,-4.692193,-3.011565,-0.140161,17.250877
VV_std,677735.0,1.275253,0.567183,2.770714e-01,0.907779,1.169121,1.496336,13.834066
VH_std,677735.0,1.290566,0.553330,3.185217e-01,0.914629,1.171169,1.516405,7.806411
texture_contrast,677735.0,2.927071,1.117046,1.124659e-01,2.130529,3.093220,3.781547,5.699752
NDVI,677735.0,0.368923,0.200327,-5.117139e-01,0.220561,0.340399,0.496545,0.903317


In [3]:
def run_lightgbm_cross_validation(X, y, features, params, n_splits=5, random_state=42):
    splitter = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )

    metric_rows = []
    importance_rows = []
    models = []

    for fold_number, (train_idx, validation_idx) in enumerate(splitter.split(X), start=1):
        started = perf_counter()
        model = LGBMRegressor(**params)
        model.fit(
            X.iloc[train_idx],
            y.iloc[train_idx],
        )
        elapsed_seconds = perf_counter() - started

        train_prediction = model.predict(X.iloc[train_idx])
        validation_prediction = model.predict(X.iloc[validation_idx])

        for subset, indices, prediction in (
            ("train", train_idx, train_prediction),
            ("validation", validation_idx, validation_prediction),
        ):
            metric_rows.append({
                "fold": fold_number,
                "subset": subset,
                "rows": len(indices),
                "n_features": len(features),
                "elapsed_seconds": elapsed_seconds,
                **calculate_metrics(y.iloc[indices], prediction),
            })

        for feature, importance in zip(features, model.feature_importances_):
            importance_rows.append({
                "fold": fold_number,
                "feature": feature,
                "importance": float(importance),
            })

        models.append(model)
        validation_metrics = metric_rows[-1]
        print(
            f"Fold {fold_number}/{n_splits} | "
            f"R2={validation_metrics['R2']:.4f} | "
            f"MAE={validation_metrics['MAE']:.4f} C | "
            f"MAPE={validation_metrics['MAPE_percent']:.4f}% | "
            f"time={elapsed_seconds:.2f}s"
        )

    return pd.DataFrame(metric_rows), pd.DataFrame(importance_rows), models


metrics_df, feature_importance_df, models = run_lightgbm_cross_validation(
    X,
    y,
    FEATURES,
    BEST_PARAMS,
    n_splits=N_SPLITS,
    random_state=RANDOM_STATE,
)

Fold 1/5 | R2=0.7555 | MAE=1.1269 C | MAPE=2.5218% | time=3.10s


Fold 2/5 | R2=0.7558 | MAE=1.1272 C | MAPE=2.5208% | time=3.13s


Fold 3/5 | R2=0.7562 | MAE=1.1241 C | MAPE=2.5141% | time=3.15s


Fold 4/5 | R2=0.7549 | MAE=1.1284 C | MAPE=2.5240% | time=3.17s


Fold 5/5 | R2=0.7551 | MAE=1.1284 C | MAPE=2.5238% | time=3.18s


In [4]:
metrics_summary_df = (
    metrics_df.groupby("subset")[["R2", "MAE", "MAPE", "MAPE_percent", "elapsed_seconds"]]
    .agg(["mean", "std", "min", "max"])
    .reset_index()
)
metrics_summary_df.columns = [
    "_".join(str(part) for part in column if part)
    if isinstance(column, tuple)
    else column
    for column in metrics_summary_df.columns
]

feature_importance_summary_df = (
    feature_importance_df.assign(
        rank=feature_importance_df.groupby("fold")["importance"].rank(
            ascending=False,
            method="average",
        )
    )
    .groupby("feature")
    .agg(
        importance_mean=("importance", "mean"),
        importance_std=("importance", "std"),
        importance_min=("importance", "min"),
        importance_max=("importance", "max"),
        mean_rank=("rank", "mean"),
    )
    .sort_values("importance_mean", ascending=False)
    .reset_index()
)

METRICS_PATH = OUTPUT_DIR / "lightgbm_5fold_metrics.csv"
SUMMARY_PATH = OUTPUT_DIR / "lightgbm_5fold_metrics_summary.csv"
IMPORTANCE_PATH = OUTPUT_DIR / "lightgbm_5fold_feature_importance.csv"
IMPORTANCE_SUMMARY_PATH = OUTPUT_DIR / "lightgbm_5fold_feature_importance_summary.csv"
PARAMETERS_PATH = OUTPUT_DIR / "lightgbm_parameters.csv"

metrics_df.to_csv(METRICS_PATH, index=False)
metrics_summary_df.to_csv(SUMMARY_PATH, index=False)
feature_importance_df.to_csv(IMPORTANCE_PATH, index=False)
feature_importance_summary_df.to_csv(IMPORTANCE_SUMMARY_PATH, index=False)
pd.DataFrame([BEST_PARAMS]).to_csv(PARAMETERS_PATH, index=False)

print("\nCSV files saved successfully:")
for output_path in (
    METRICS_PATH,
    SUMMARY_PATH,
    IMPORTANCE_PATH,
    IMPORTANCE_SUMMARY_PATH,
    PARAMETERS_PATH,
):
    print(" -", output_path.resolve())

print("\nValidation metrics by fold:")
display(metrics_df.query("subset == 'validation'")[["fold", "rows", "R2", "MAE", "MAPE_percent"]])

print("\nCross-validation summary:")
display(metrics_summary_df)

print("\nTop 20 LightGBM key drivers:")
display(feature_importance_summary_df.head(20))


CSV files saved successfully:
 - /Users/simon/Downloads/earth_obs/urban_heat_island_analysis_EODA/model_outputs/lightgbm_5fold_cv/lightgbm_5fold_metrics.csv
 - /Users/simon/Downloads/earth_obs/urban_heat_island_analysis_EODA/model_outputs/lightgbm_5fold_cv/lightgbm_5fold_metrics_summary.csv
 - /Users/simon/Downloads/earth_obs/urban_heat_island_analysis_EODA/model_outputs/lightgbm_5fold_cv/lightgbm_5fold_feature_importance.csv
 - /Users/simon/Downloads/earth_obs/urban_heat_island_analysis_EODA/model_outputs/lightgbm_5fold_cv/lightgbm_5fold_feature_importance_summary.csv
 - /Users/simon/Downloads/earth_obs/urban_heat_island_analysis_EODA/model_outputs/lightgbm_5fold_cv/lightgbm_parameters.csv

Validation metrics by fold:


,fold,rows,R2,MAE,MAPE_percent
1,1,135547,0.755514,1.126872,2.521822
3,2,135547,0.755798,1.127156,2.520796
5,3,135547,0.756151,1.124147,2.514122
7,4,135547,0.754873,1.128443,2.523977
9,5,135547,0.755108,1.128449,2.523800



Cross-validation summary:


,subset,R2_mean,R2_std,R2_min,R2_max,MAE_mean,MAE_std,MAE_min,MAE_max,MAPE_mean,...,MAPE_min,MAPE_max,MAPE_percent_mean,MAPE_percent_std,MAPE_percent_min,MAPE_percent_max,elapsed_seconds_mean,elapsed_seconds_std,elapsed_seconds_min,elapsed_seconds_max
0,train,0.757669,0.000139,0.757510,0.757868,1.122449,0.000434,1.121937,1.122965,0.025107,...,0.025097,0.025116,2.510745,0.000865,2.509707,2.511611,3.145562,0.031529,3.098438,3.176702
1,validation,0.755489,0.000515,0.754873,0.756151,1.127013,0.001758,1.124147,1.128449,0.025209,...,0.025141,0.025240,2.520903,0.004021,2.514122,2.523977,3.145562,0.031529,3.098438,3.176702



Top 20 LightGBM key drivers:


,feature,importance_mean,importance_std,importance_min,importance_max,mean_rank
0,elevation_focal,736.8,13.590438,724.0,751.0,1.2
1,NDWI_focal,723.0,5.958188,715.0,731.0,1.8
2,NDBI_focal,680.6,6.985700,674.0,692.0,3.0
3,VH_focal,662.6,8.111720,654.0,670.0,4.0
4,VV_focal,538.6,13.794927,519.0,557.0,5.0
5,Albedo_focal,513.2,9.628084,502.0,524.0,6.0
6,NDVI_focal,411.6,8.734987,400.0,422.0,7.0
7,NLI,331.0,21.047565,304.0,358.0,8.0
8,texture_contrast_focal,275.0,10.464225,267.0,293.0,9.3
9,PIS_focal,271.6,13.011533,256.0,288.0,9.7


## Output files

After execution, the notebook creates:

- `model_outputs/lightgbm_5fold_cv/lightgbm_5fold_metrics.csv`: train and validation metrics for every fold;
- `model_outputs/lightgbm_5fold_cv/lightgbm_5fold_metrics_summary.csv`: mean, standard deviation, minimum and maximum metrics;
- `model_outputs/lightgbm_5fold_cv/lightgbm_5fold_feature_importance.csv`: feature importance for every fold;
- `model_outputs/lightgbm_5fold_cv/lightgbm_5fold_feature_importance_summary.csv`: stable key-driver ranking;
- `model_outputs/lightgbm_5fold_cv/lightgbm_parameters.csv`: exact model parameters used.